In [ ]:
# Example 5: Advanced Queue and Producer-Consumer Pattern
# Demonstrate multiprocess Queue with producer-consumer pattern

import multiprocess as mp
from multiprocess import Queue, Process
import time
import random

def producer(queue, producer_id, num_items):
    """Producer function that generates data and puts it in queue"""
    for i in range(num_items):
        # Simulate work to generate data
        time.sleep(random.uniform(0.05, 0.15))
        
        # Create data item
        data = {
            'producer_id': producer_id,
            'item_id': i,
            'value': random.randint(1, 100),
            'timestamp': time.time()
        }
        
        queue.put(data)
        print(f"Producer {producer_id}: Generated item {i} with value {data['value']}")
    
    # Signal completion
    queue.put(None)
    print(f"Producer {producer_id}: Finished generating {num_items} items")

def consumer(queue, consumer_id):
    """Consumer function that processes data from queue"""
    processed_count = 0
    total_value = 0
    
    while True:
        try:
            # Get item from queue with timeout
            item = queue.get(timeout=2.0)
            
            if item is None:
                print(f"Consumer {consumer_id}: Received termination signal")
                break
            
            # Process the item
            processed_count += 1
            total_value += item['value']
            print(f"Consumer {consumer_id}: Processed item {item['item_id']} "
                  f"from producer {item['producer_id']} (value: {item['value']})")
            
            # Simulate processing time
            time.sleep(random.uniform(0.02, 0.08))
            
        except Exception as e:
            print(f"Consumer {consumer_id}: Queue timeout or error: {e}")
            break
    
    print(f"Consumer {consumer_id}: Processed {processed_count} items, "
          f"total value: {total_value}")
    return processed_count, total_value

if __name__ == '__main__' or True:  # Allow execution in Jupyter
    # Create shared queue
    task_queue = Queue(maxsize=10)
    
    # Create producer processes
    producers = []
    for i in range(2):
        p = Process(target=producer, args=(task_queue, i, 3))
        producers.append(p)
        p.start()
    
    # Create consumer processes
    consumers = []
    for i in range(2):
        p = Process(target=consumer, args=(task_queue, i))
        consumers.append(p)
        p.start()
    
    # Wait for producers to finish
    for p in producers:
        p.join()
    
    # Wait for consumers to finish
    for p in consumers:
        p.join()
    
    print("Example 5: Producer-Consumer pattern completed successfully!")

In [ ]:
# Example 4: Custom Manager and Shared Objects
# Demonstrate multiprocess Manager for shared state between processes

import multiprocess as mp
from multiprocess import Manager
import time
import random

def worker_with_shared_state(worker_id, shared_dict, shared_list, lock):
    """Worker function that modifies shared objects safely"""
    for i in range(3):
        # Simulate some work
        time.sleep(random.uniform(0.1, 0.3))
        
        # Use lock to ensure thread-safe operations
        with lock:
            # Update shared dictionary
            shared_dict[f'worker_{worker_id}_step_{i}'] = f'completed at {time.time():.2f}'
            shared_list.append(f'Worker {worker_id} completed step {i}')
            print(f"Worker {worker_id}: Step {i} completed")

if __name__ == '__main__' or True:  # Allow execution in Jupyter
    # Create manager for shared objects
    with Manager() as manager:
        shared_dict = manager.dict()
        shared_list = manager.list()
        lock = manager.Lock()
        
        # Create and start worker processes
        processes = []
        for worker_id in range(3):
            p = mp.Process(target=worker_with_shared_state, 
                          args=(worker_id, shared_dict, shared_list, lock))
            processes.append(p)
            p.start()
        
        # Wait for all processes to complete
        for p in processes:
            p.join()
        
        # Display results
        print(f"\nShared dictionary has {len(shared_dict)} entries:")
        for key, value in sorted(shared_dict.items()):
            print(f"  {key}: {value}")
        
        print(f"\nShared list has {len(shared_list)} entries:")
        for item in shared_list:
            print(f"  {item}")
            
print("Example 4: Custom Manager and Shared Objects completed successfully!")

In [ ]:
# Basic parallel processing with Pool and Process classes
import multiprocess as mp
import time
import numpy as np
from typing import List, Callable, Any, Dict

print("=== Multiprocess Basic Parallel Processing ===")

def cpu_intensive_task(n: int) -> Dict[str, Any]:
    """CPU-intensive task for demonstrating parallel processing"""
    start_time = time.time()
    
    # Simulate CPU-intensive computation
    result = 0
    for i in range(n * 100000):
        result += i ** 2
    
    # Calculate some statistics
    numbers = np.random.rand(n * 1000)
    stats = {
        'task_id': n,
        'computation_result': result % 1000000,  # Keep result manageable
        'mean': float(np.mean(numbers)),
        'std': float(np.std(numbers)),
        'processing_time': time.time() - start_time,
        'data_points': len(numbers)
    }
    
    return stats

def basic_pool_processing():
    """Demonstrate basic Pool-based parallel processing"""
    
    # Create tasks of varying complexity
    tasks = [10, 15, 20, 25, 30, 35, 40, 45]
    
    # Sequential processing baseline
    print("Running sequential processing...")
    start_time = time.time()
    sequential_results = [cpu_intensive_task(task) for task in tasks]
    sequential_time = time.time() - start_time
    
    # Parallel processing with Pool
    print("Running parallel processing...")
    start_time = time.time()
    
    with mp.Pool(processes=4) as pool:
        parallel_results = pool.map(cpu_intensive_task, tasks)
    
    parallel_time = time.time() - start_time
    
    # Compare results
    speedup = sequential_time / parallel_time
    
    return {
        'tasks_count': len(tasks),
        'sequential_time': sequential_time,
        'parallel_time': parallel_time,
        'speedup': speedup,
        'sequential_results': sequential_results,
        'parallel_results': parallel_results
    }

def process_class_demo():
    """Demonstrate Process class usage"""
    
    def worker_function(worker_id: int, shared_queue: mp.Queue, work_items: List[int]):
        """Worker function that processes items and puts results in shared queue"""
        results = []
        
        for item in work_items:
            # Process the item
            processed = item * item + worker_id
            results.append({
                'worker_id': worker_id,
                'input': item,
                'output': processed,
                'timestamp': time.time()
            })
        
        # Put results in shared queue
        shared_queue.put({
            'worker_id': worker_id,
            'results': results,
            'items_processed': len(work_items)
        })
    
    # Create shared queue
    result_queue = mp.Queue()
    
    # Divide work among processes
    work_data = list(range(1, 21))  # 20 items total
    chunk_size = 5
    work_chunks = [work_data[i:i + chunk_size] for i in range(0, len(work_data), chunk_size)]
    
    # Create and start processes
    processes = []
    start_time = time.time()
    
    for i, chunk in enumerate(work_chunks):
        process = mp.Process(target=worker_function, args=(i, result_queue, chunk))
        processes.append(process)
        process.start()
    
    # Wait for all processes to complete
    for process in processes:
        process.join()
    
    processing_time = time.time() - start_time
    
    # Collect results from queue
    all_results = []
    while not result_queue.empty():
        result = result_queue.get()
        all_results.append(result)
    
    return {
        'processes_count': len(processes),
        'work_chunks': len(work_chunks),
        'total_items': len(work_data),
        'processing_time': processing_time,
        'results': all_results
    }

# Run basic parallel processing demonstrations
print("Testing Pool-based parallel processing...")
pool_results = basic_pool_processing()

print(f"\nPool Processing Results:")
print(f"Tasks processed: {pool_results['tasks_count']}")
print(f"Sequential time: {pool_results['sequential_time']:.3f}s")
print(f"Parallel time: {pool_results['parallel_time']:.3f}s")
print(f"Speedup achieved: {pool_results['speedup']:.2f}x")

# Verify results consistency
sequential_means = [r['mean'] for r in pool_results['sequential_results']]
parallel_means = [r['mean'] for r in pool_results['parallel_results']]
results_consistent = all(abs(s - p) < 1e-10 for s, p in zip(sequential_means, parallel_means))
print(f"Results consistent: {'✓' if results_consistent else '✗'}")

print(f"\nTesting Process class functionality...")
process_results = process_class_demo()

print(f"Process Class Results:")
print(f"Processes used: {process_results['processes_count']}")
print(f"Work chunks: {process_results['work_chunks']}")
print(f"Total items processed: {process_results['total_items']}")
print(f"Processing time: {process_results['processing_time']:.3f}s")

# Verify all items were processed
total_processed = sum(r['items_processed'] for r in process_results['results'])
print(f"Items processed verification: {total_processed}/{process_results['total_items']} ({'✓' if total_processed == process_results['total_items'] else '✗'})")

print("Basic parallel processing completed successfully!")

In [ ]:
# Advanced serialization capabilities with complex functions and objects
from functools import partial
import dill

print("=== Multiprocess Advanced Serialization ===")

class ComplexDataProcessor:
    """Complex class to demonstrate advanced serialization"""
    
    def __init__(self, multiplier: float = 2.0, offset: float = 1.0):
        self.multiplier = multiplier
        self.offset = offset
        self.processed_count = 0
    
    def process_data(self, data: List[float]) -> Dict[str, Any]:
        """Process data with instance variables"""
        processed = [(x * self.multiplier) + self.offset for x in data]
        self.processed_count += len(data)
        
        return {
            'original_data': data,
            'processed_data': processed,
            'multiplier': self.multiplier,
            'offset': self.offset,
            'count': len(data),
            'mean_processed': np.mean(processed)
        }

def test_lambda_serialization():
    """Test serialization of lambda functions (impossible with standard multiprocessing)"""
    
    # Lambda functions that standard multiprocessing cannot serialize
    lambda_functions = [
        lambda x: x ** 2,
        lambda x: np.sin(x) * 10,
        lambda x: x * 3 + 7,
        lambda x: np.exp(-x) if x > 0 else 0
    ]
    
    test_data = [np.random.rand(100) for _ in range(len(lambda_functions))]
    
    results = []
    
    # Apply each lambda function to corresponding data using multiprocess
    with mp.Pool(processes=2) as pool:
        for i, (func, data) in enumerate(zip(lambda_functions, test_data)):
            try:
                # This works with multiprocess but fails with standard multiprocessing
                result = pool.map(func, data)
                results.append({
                    'function_id': i,
                    'data_points': len(data),
                    'result_sample': result[:5],  # First 5 results
                    'serialization_success': True
                })
            except Exception as e:
                results.append({
                    'function_id': i,
                    'error': str(e),
                    'serialization_success': False
                })
    
    return results

def test_closure_serialization():
    """Test serialization of closures (functions that capture local variables)"""
    
    def create_processor(base_value: float, operation: str):
        """Factory function that creates closures"""
        
        if operation == 'multiply':
            def processor(x):
                return x * base_value
        elif operation == 'add':
            def processor(x):
                return x + base_value
        elif operation == 'power':
            def processor(x):
                return x ** base_value
        else:
            def processor(x):
                return x
        
        return processor
    
    # Create closures with different captured variables
    processors = [
        create_processor(2.5, 'multiply'),
        create_processor(10.0, 'add'),
        create_processor(1.5, 'power')
    ]
    
    test_values = [1.0, 2.0, 3.0, 4.0, 5.0]
    
    results = []
    
    with mp.Pool(processes=2) as pool:
        for i, processor in enumerate(processors):
            result = pool.map(processor, test_values)
            results.append({
                'processor_id': i,
                'test_values': test_values,
                'results': result,
                'closure_serialized': True
            })
    
    return results

def test_method_serialization():
    """Test serialization of instance methods"""
    
    # Create instances with different configurations
    processors = [
        ComplexDataProcessor(multiplier=2.0, offset=0.0),
        ComplexDataProcessor(multiplier=1.5, offset=5.0),
        ComplexDataProcessor(multiplier=3.0, offset=-2.0)
    ]
    
    # Test data for each processor
    test_datasets = [
        [1.0, 2.0, 3.0, 4.0, 5.0],
        [10.0, 20.0, 30.0],
        [0.1, 0.2, 0.3, 0.4]
    ]
    
    results = []
    
    # Use partial to create callable that can be serialized
    for i, (processor, dataset) in enumerate(zip(processors, test_datasets)):
        # Create a wrapper function that can be serialized
        def process_with_instance(data_and_processor):
            data, proc = data_and_processor
            return proc.process_data(data)
        
        # Use pool to process
        with mp.Pool(processes=1) as pool:
            # Prepare data with processor instance
            data_with_processor = (dataset, processor)
            result = process_with_instance(data_with_processor)
            
            results.append({
                'processor_id': i,
                'multiplier': processor.multiplier,
                'offset': processor.offset,
                'input_data': dataset,
                'result': result,
                'method_serialized': True
            })
    
    return results

# Test advanced serialization capabilities
print("Testing lambda function serialization...")
lambda_results = test_lambda_serialization()

print("Lambda Function Serialization Results:")
successful_lambdas = sum(1 for r in lambda_results if r.get('serialization_success', False))
print(f"Lambda functions successfully serialized: {successful_lambdas}/{len(lambda_results)}")

for result in lambda_results:
    if result.get('serialization_success'):
        print(f"  Function {result['function_id']}: ✓ Processed {result['data_points']} points")
        print(f"    Sample results: {result['result_sample']}")
    else:
        raise AssertionError(f"Function {result['function_id']}: multiprocess could not serialize/run the lambda - {result.get('error', 'Unknown error')}")

print(f"\nTesting closure serialization...")
closure_results = test_closure_serialization()

print("Closure Serialization Results:")
for result in closure_results:
    print(f"  Processor {result['processor_id']}: ✓ Closure serialized successfully")
    print(f"    Input: {result['test_values']}")
    print(f"    Output: {result['results']}")

print(f"\nTesting method serialization...")
method_results = test_method_serialization()

print("Method Serialization Results:")
for result in method_results:
    print(f"  Processor {result['processor_id']}: ✓ Method serialized successfully")
    print(f"    Config: multiplier={result['multiplier']}, offset={result['offset']}")
    print(f"    Mean processed: {result['result']['mean_processed']:.2f}")

print("Advanced serialization capabilities completed successfully!")

In [ ]:
# Scientific computing and data processing pipeline optimization
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
import psutil

print("=== Multiprocess Scientific Computing Pipeline ===")

def scientific_computation_task(dataset_config: Dict[str, Any]) -> Dict[str, Any]:
    """Scientific computation task that benefits from parallel processing"""
    
    dataset_id = dataset_config['id']
    size = dataset_config['size']
    computation_type = dataset_config['type']
    
    start_time = time.time()
    
    # Generate synthetic scientific data
    if computation_type == 'matrix_operations':
        # Matrix operations
        matrix_a = np.random.randn(size, size)
        matrix_b = np.random.randn(size, size)
        
        # Perform computations
        result = np.dot(matrix_a, matrix_b)
        eigenvalues = np.linalg.eigvals(result[:min(100, size), :min(100, size)])  # Limit for performance
        
        computed_results = {
            'determinant': np.linalg.det(result[:10, :10]),  # Small submatrix for det
            'trace': np.trace(result),
            'max_eigenvalue': np.max(np.real(eigenvalues)),
            'condition_number': np.linalg.cond(matrix_a[:50, :50])  # Limit for performance
        }
        
    elif computation_type == 'statistical_analysis':
        # Statistical analysis
        data = np.random.exponential(2.0, size * 100)
        
        # Compute various statistics
        computed_results = {
            'mean': np.mean(data),
            'median': np.median(data),
            'std': np.std(data),
            'skewness': float(pd.Series(data).skew()),
            'kurtosis': float(pd.Series(data).kurtosis()),
            'percentiles': np.percentile(data, [25, 50, 75, 90, 95, 99]).tolist()
        }
        
    elif computation_type == 'signal_processing':
        # Signal processing
        signal = np.random.randn(size * 1000)
        freqs = np.fft.fftfreq(len(signal))
        fft_signal = np.fft.fft(signal)
        
        computed_results = {
            'signal_power': np.sum(signal ** 2),
            'dominant_frequency': freqs[np.argmax(np.abs(fft_signal))],
            'signal_energy': np.sum(np.abs(fft_signal) ** 2),
            'snr_estimate': 10 * np.log10(np.var(signal) / (np.var(signal) * 0.1 + 1e-10))
        }
        
    else:
        # Default computation
        data = np.random.rand(size * 100)
        computed_results = {
            'sum': np.sum(data),
            'product': np.prod(data[:100]),  # Limit to prevent overflow
            'variance': np.var(data)
        }
    
    processing_time = time.time() - start_time
    
    return {
        'dataset_id': dataset_id,
        'computation_type': computation_type,
        'data_size': size,
        'processing_time': processing_time,
        'results': computed_results,
        'memory_used_mb': psutil.Process().memory_info().rss / (1024 * 1024)
    }

def create_scientific_pipeline():
    """Create a scientific computing pipeline with multiple computation types"""
    
    # Define different types of scientific computations
    computation_configs = [
        {'id': f'matrix_{i}', 'size': 200 + i * 50, 'type': 'matrix_operations'} for i in range(3)
    ] + [
        {'id': f'stats_{i}', 'size': 500 + i * 200, 'type': 'statistical_analysis'} for i in range(3)
    ] + [
        {'id': f'signal_{i}', 'size': 100 + i * 50, 'type': 'signal_processing'} for i in range(3)
    ]
    
    # Sequential processing baseline
    print("Running sequential scientific computing pipeline...")
    start_time = time.time()
    sequential_results = [scientific_computation_task(config) for config in computation_configs]
    sequential_time = time.time() - start_time
    
    # Parallel processing with multiprocess
    print("Running parallel scientific computing pipeline...")
    start_time = time.time()
    
    with mp.Pool(processes=4) as pool:
        parallel_results = pool.map(scientific_computation_task, computation_configs)
    
    parallel_time = time.time() - start_time
    
    # Analyze results
    speedup = sequential_time / parallel_time
    total_processing_time_seq = sum(r['processing_time'] for r in sequential_results)
    total_processing_time_par = sum(r['processing_time'] for r in parallel_results)
    
    return {
        'computation_configs': computation_configs,
        'sequential_results': sequential_results,
        'parallel_results': parallel_results,
        'sequential_wall_time': sequential_time,
        'parallel_wall_time': parallel_time,
        'sequential_cpu_time': total_processing_time_seq,
        'parallel_cpu_time': total_processing_time_par,
        'speedup': speedup,
        'tasks_count': len(computation_configs)
    }

def data_processing_pipeline():
    """Demonstrate data processing pipeline with multiprocess"""
    
    def process_data_chunk(chunk_info: Dict[str, Any]) -> Dict[str, Any]:
        """Process a chunk of data"""
        
        chunk_id = chunk_info['chunk_id']
        data_size = chunk_info['data_size']
        operations = chunk_info['operations']
        
        # Generate synthetic data
        data = np.random.randn(data_size, 10)  # 10 features
        
        results = {'chunk_id': chunk_id, 'data_shape': data.shape}
        
        start_time = time.time()
        
        # Apply requested operations
        if 'normalize' in operations:
            data = (data - np.mean(data, axis=0)) / np.std(data, axis=0)
            results['normalized'] = True
        
        if 'pca' in operations:
            # Simple PCA approximation
            cov_matrix = np.cov(data.T)
            eigenvals, _ = np.linalg.eigh(cov_matrix)
            results['explained_variance'] = eigenvals[-3:].tolist()  # Top 3 components
        
        if 'cluster' in operations:
            # Simple k-means approximation (2 clusters)
            centroids = data[np.random.choice(data.shape[0], 2, replace=False)]
            distances = np.linalg.norm(data[:, np.newaxis] - centroids, axis=2)
            assignments = np.argmin(distances, axis=1)
            results['cluster_sizes'] = [np.sum(assignments == 0), np.sum(assignments == 1)]
        
        results['processing_time'] = time.time() - start_time
        
        return results
    
    # Create data processing tasks
    data_chunks = [
        {'chunk_id': i, 'data_size': 1000 + i * 200, 'operations': ['normalize', 'pca']}
        for i in range(5)
    ] + [
        {'chunk_id': i + 5, 'data_size': 800 + i * 150, 'operations': ['normalize', 'cluster']}
        for i in range(3)
    ]
    
    # Process chunks in parallel
    with mp.Pool(processes=3) as pool:
        processing_results = pool.map(process_data_chunk, data_chunks)
    
    return {
        'chunks_processed': len(data_chunks),
        'results': processing_results,
        'total_data_points': sum(chunk['data_size'] for chunk in data_chunks)
    }

# Run scientific computing pipeline
print("Executing scientific computing pipeline...")
pipeline_results = create_scientific_pipeline()

print(f"\nScientific Computing Pipeline Results:")
print(f"Tasks executed: {pipeline_results['tasks_count']}")
print(f"Sequential wall time: {pipeline_results['sequential_wall_time']:.3f}s")
print(f"Parallel wall time: {pipeline_results['parallel_wall_time']:.3f}s")
print(f"Speedup achieved: {pipeline_results['speedup']:.2f}x")
print(f"CPU time savings: {pipeline_results['sequential_cpu_time'] - pipeline_results['parallel_cpu_time']:.2f}s")

# Show sample results by computation type
computation_types = {}
for result in pipeline_results['parallel_results']:
    comp_type = result['computation_type']
    if comp_type not in computation_types:
        computation_types[comp_type] = []
    computation_types[comp_type].append(result['processing_time'])

print(f"\nProcessing times by computation type:")
for comp_type, times in computation_types.items():
    avg_time = np.mean(times)
    print(f"  {comp_type}: {avg_time:.3f}s avg ({len(times)} tasks)")

# Run data processing pipeline
print(f"\nExecuting data processing pipeline...")
data_pipeline_results = data_processing_pipeline()

print(f"Data Processing Pipeline Results:")
print(f"Chunks processed: {data_pipeline_results['chunks_processed']}")
print(f"Total data points: {data_pipeline_results['total_data_points']:,}")

# Show processing time statistics
processing_times = [r['processing_time'] for r in data_pipeline_results['results']]
print(f"Average chunk processing time: {np.mean(processing_times):.3f}s")
print(f"Total processing time: {sum(processing_times):.3f}s")

print("Scientific computing and data processing pipeline completed successfully!")